# שלב 2 — Spatial Encoding (H3)

ממירים כל מסלול מרצף נקודות GPS מדויקות לרצף **תאי H3**. כך מתארים מסלול כ"מעבר בין תאים" — מה שמייעל מאוד את שלב 3 (השוואת מחרוזות תאים במקום קואורדינטות).

**למה H3?** משושים עם שכנוּת אחידה (מרחק זהה לכל שכן — נוח לעקיבת מסלולים), היררכיה (אפשר לעבור רזולוציות), וספרייה פייתונית מצוינת.

**הקלט:** ה-Parquet משלב 1 (`out/trips_clean_sample.parquet`) — מכיל גם עמודות פיצ'רים ל-ML (winsorize/log/encoding/class_weight) שלא רלוונטיות כאן; שלב זה שואב ממנו רק `TRIP_ID/TAXI_ID/points/n_points/duration_sec/distance_km/avg_kmh`. **הפלט:** טבלה עם עמודת `cells` (רצף תאי H3, כפילויות עוקבות מכווצות) — הקלט לשלב 3.

> ה-encoding נעשה ב-`pandas_udf` (וקטורי, רץ פר-partition) כי H3 היא ספריית פייתון ואין פונקציית Spark נייטיבית לזה.


## 1. SparkSession

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, StringType
import pandas as pd
import h3

spark = (
    SparkSession.builder
    .appName("porto-taxi-stage2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("h3 version:", h3.__version__)


## 2. קונפיגורציה

In [ ]:
USE_SAMPLE = True

if USE_SAMPLE:
    INPUT_PATH  = "out/trips_clean_sample.parquet"
    OUTPUT_PATH = "out/trips_h3_sample.parquet"
else:
    INPUT_PATH  = "gs://YOUR_BUCKET/clean/trips_clean.parquet"
    OUTPUT_PATH = "gs://YOUR_BUCKET/encoded/trips_h3.parquet"

RESOLUTION = 9   # נבחר אחרי ניסוי הרזולוציות בהמשך


In [ ]:
trips = spark.read.parquet(INPUT_PATH)
print("trips:", trips.count())
trips.printSchema()


## 3. ה-Encoder (factory לפי רזולוציה)

לכל נקודה `p = [lon, lat]` נחשב תא H3 (`h3.latlng_to_cell(lat, lon, res)`), ונכווץ **כפילויות עוקבות** — כך שמסלול הופך לרצף התאים שהוא חוצה בפועל.

In [ ]:
def make_h3_encoder(res: int):
    @pandas_udf(ArrayType(StringType()))
    def _encode(points_series: pd.Series) -> pd.Series:
        def conv(points):
            if points is None:
                return []
            seq, prev = [], None
            for p in points:
                c = h3.latlng_to_cell(p[1], p[0], res)  # (lat, lon, res)
                if c != prev:
                    seq.append(c)
                    prev = c
            return seq
        return points_series.apply(conv)
    return _encode


## 4. ניסוי רזולוציות (8 / 9 / 10)

ה-trade-off מהמסמך: תאים גדולים מדי → לא מבחינים בין כבישים סמוכים; תאים קטנים מדי → בזבוז משאבים ורצפים ארוכים.

נשווה לכל רזולוציה: אורך רצף ממוצע, סך תאים ייחודיים, ויחס הכיווץ (נקודות→תאים). גודל קצה משוער: res 8 ≈ 460מ', res 9 ≈ 175מ', res 10 ≈ 65מ'.

In [ ]:
sample_for_exp = trips.select("TRIP_ID", "points", "n_points").limit(3000).cache()

rows = []
for res in [8, 9, 10]:
    enc = sample_for_exp.withColumn("cells", make_h3_encoder(res)("points")) \
                        .withColumn("n_cells", F.size("cells"))
    agg = enc.agg(
        F.avg("n_cells").alias("avg_cells"),
        F.avg("n_points").alias("avg_points"),
    ).first()
    distinct_cells = enc.select(F.explode("cells").alias("c")).select(F.countDistinct("c")).first()[0]
    rows.append((res, round(agg["avg_points"],1), round(agg["avg_cells"],1),
                 round(agg["avg_points"]/agg["avg_cells"],2), distinct_cells))

print(f"{'res':>4} {'avg_pts':>8} {'avg_cells':>10} {'compress':>9} {'distinct_cells':>15}")
for r in rows:
    print(f"{r[0]:>4} {r[1]:>8} {r[2]:>10} {r[3]:>9} {r[4]:>15}")


**איך לבחור:** רוצים מספיק תאים שונים כדי להבחין בין כבישים סמוכים, אבל לא רצפים ארוכים מדי. res 9 (~175מ') הוא בדרך כלל איזון טוב לעיר; res 10 מדויק יותר אך יקר. עדכנו `RESOLUTION` למעלה לפי התוצאות.

## 5. Encoding סופי + שמירה

In [ ]:
encoded = (
    trips
    .withColumn("cells", make_h3_encoder(RESOLUTION)("points"))
    .withColumn("n_cells", F.size("cells"))
    .filter(F.col("n_cells") >= 2)            # צריך לפחות מעבר אחד בין תאים
)

out = encoded.select(
    "TRIP_ID", "TAXI_ID", "cells", "n_cells",
    "n_points", "duration_sec", "distance_km", "avg_kmh"
)

out.write.mode("overwrite").parquet(OUTPUT_PATH)
print("saved:", OUTPUT_PATH)
out.select("TRIP_ID", "n_cells", "distance_km").show(5)
out.select(F.avg("n_cells").alias("avg_n_cells"),
           F.expr("approx_percentile(n_cells, array(0.5,0.9,0.99))").alias("n_cells_p50_p90_p99")).show(truncate=False)


## 6. בונוס — מוקדי פעילות (Activity Hotspots)

מחברים ישירות למטרה "אילו אזורים הם מוקדי פעילות": סופרים כמה נסיעות (וכמה מוניות ייחודיות) עוברות בכל תא. `approx_count_distinct` הוא מקורב, זול ומבוזר.

In [ ]:
hot = (
    out.select(F.explode("cells").alias("cell"), "TRIP_ID", "TAXI_ID")
       .groupBy("cell")
       .agg(F.countDistinct("TRIP_ID").alias("n_trips"),
            F.approx_count_distinct("TAXI_ID").alias("approx_n_taxis"))
       .orderBy(F.desc("n_trips"))
)
hot.show(15, truncate=False)


## הערות להרצה על DataProc
- `USE_SAMPLE = False`, נתיבי `gs://`, הסירו `.master("local[*]")`, העלו `shuffle.partitions`.
- ודאו ש-`h3` מותקן על ה-workers: ב-Dataproc אפשר עם `--metadata PIP_PACKAGES=h3` + initialization action `pip-install`, או image עם h3 מותקן.

**תוצר:** `trips_h3.parquet` עם עמודת `cells` — הקלט לשלב 3.
